In [ ]:
#| default_exp nbskill

## Installing the skill

Install the notebook workflow instructions, then register reusable local repositories.

The repository ships instructions that teach agents how to use the Python notebook workflow. This installer copies the package-owned skill and references into a requested skills directory.

The package ships a compact routing skill. The operational guidance lives in the Python skill module, so installation copies package-owned instructions rather than generating a second workflow.

### Production contract
Skill installation is Python-first. It installs only into the requested Codex or Claude skills directory, optionally installs notebook hooks, and keeps reference indexing separate from normal workflow use.

In [ ]:
from nbskill.foundation import demo_path, remove_demo_path

In [ ]:
#| export
import json, subprocess, tomllib
from importlib.resources import files
from pathlib import Path

from nbskill.foundation import install_nbdev_pre_commit_hooks

### Installing agent instructions
The installer copies the compact routing skill and its reference material. Native notebook methodology lives in the Python skill module; installation only makes those files available to the chosen agent.

In [ ]:
#| exporti
def _write_if_changed(path, text):
    old = path.read_text(encoding="utf-8") if path.exists() else ""
    if old == text: return False
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return True

In [ ]:
#| exporti
def _install_skill_tree(root, skill_name, skill_text, references, overwrite=True):
    """Copy changed skill files and leave byte-identical files untouched."""
    dst_dir = root / skill_name
    files_to_install = [(dst_dir / "SKILL.md", skill_text)]
    if references.is_dir():
        files_to_install += [
            (dst_dir / "references" / ref.name, ref.read_text(encoding="utf-8"))
            for ref in references.iterdir() if ref.is_file()
        ]
    changed, unchanged = [], []
    for path,text in files_to_install:
        if path.exists() and path.read_text(encoding="utf-8") != text and not overwrite:
            raise FileExistsError(path)
        if _write_if_changed(path, text): changed.append(path)
        else: unchanged.append(path)
    return changed, unchanged

In [ ]:
#| exporti
def _skill_roots(target, skills_dir):
    target = target.lower()
    if target == "cursor": return target, []
    if skills_dir: return target, [Path(skills_dir).expanduser()]
    if target == "codex": return target, [Path.home() / ".codex" / "skills"]
    if target in {"claude", "claude-code", "claude_code"}: return target, [Path.home() / ".claude" / "skills"]
    if target == "both": return target, [Path.home() / ".codex" / "skills", Path.home() / ".claude" / "skills"]
    raise ValueError("target must be codex, claude, cursor, both, or use skills_dir")

In [ ]:
#| exporti
def _install_skill_files(roots, skill_name, overwrite):
    package = files("nbskill")
    skill_text = package.joinpath("SKILL.md").read_text(encoding="utf-8")
    references = package.joinpath("references")
    installed, unchanged = [], []
    for root in roots:
        changed, current = _install_skill_tree(root, skill_name, skill_text, references, overwrite)
        installed += changed
        unchanged += current
    return installed, unchanged

In [ ]:
#| export
def install_nbskill(
    target: str = "codex",  # codex, claude, cursor, both, or custom when skills_dir is set
    skills_dir: str | None = None,  # Parent skills directory; skill is installed below jupyter-notebooks
    skill_name: str = "jupyter-notebooks",  # Skill folder name
    overwrite: bool = True,  # Update stale managed files
    install_hooks: bool = False,  # Install nbdev-clean/nbdev-test pre-commit hooks in the current repo
    reference_roots: str = "~/projects",  # Local Git repositories to add to the reference index
    index_references: bool = True,  # Index references during installation
):
    """Install current Python workflow instructions."""
    target, roots = _skill_roots(target, skills_dir)
    installed, unchanged = _install_skill_files(roots, skill_name, overwrite)
    hooks = install_nbdev_pre_commit_hooks(Path.cwd()) if install_hooks else {"installed": False, "reason": "disabled"}
    from nbskill.knowledge import reference_discover
    reference_index = reference_discover(roots=reference_roots, ingest=index_references)
    for path in installed: print(f"Installed {path}")
    if hooks.get("installed"): print(f"Installed nbdev pre-commit hook at {hooks['hook']}")
    return {"installed": installed, "unchanged": unchanged, "hooks": hooks, "references": reference_index}

In [ ]:
#| export
def install_nbskill_cli():
    "Install notebook workflow instructions from the command line."
    import argparse
    parser = argparse.ArgumentParser(description="Install nbskill Python workflow instructions.")
    parser.add_argument("--target", default="codex")
    parser.add_argument("--skills-dir")
    parser.add_argument("--skill-name", default="jupyter-notebooks")
    parser.add_argument("--no-overwrite", dest="overwrite", action="store_false")
    parser.add_argument("--install-hooks", action="store_true")
    parser.add_argument("--reference-roots", default="~/projects")
    parser.add_argument("--no-index-references", dest="index_references", action="store_false")
    install_nbskill(**vars(parser.parse_args()))

In [ ]:
#| hide
from unittest.mock import patch

In [ ]:
#| hide
#| eval: false
install_root = demo_path("06_skill_install")
try:
    with patch("nbskill.knowledge.reference_discover", return_value={}):
        first = install_nbskill(skills_dir=str(install_root), index_references=False)
    skill_dir = install_root / "jupyter-notebooks"
    assert first["installed"]
    assert (skill_dir / "SKILL.md").exists()
    assert (skill_dir / "references" / "conversion.md").exists()
    assert (skill_dir / "references" / "extended-tools.md").exists()
    installed_text = {path: path.read_bytes() for path in skill_dir.rglob("*") if path.is_file()}
    with patch("nbskill.knowledge.reference_discover", return_value={}):
        second = install_nbskill(skills_dir=str(install_root), index_references=False)
    assert not second["installed"]
    assert second["unchanged"]
    assert installed_text == {path: path.read_bytes() for path in skill_dir.rglob("*") if path.is_file()}
finally:
    remove_demo_path(install_root)

In [ ]:
#| hide
#| eval: false
hook_root = demo_path("06_skill_hooks")
try:
    hook_root.mkdir()
    (hook_root / "nbs").mkdir()
    subprocess.run(["git", "init"], cwd=hook_root, check=True, capture_output=True)
    hook_result = install_nbdev_pre_commit_hooks(hook_root, run_nbdev_install_hooks=False)
    pre_commit = hook_root / ".git" / "hooks" / "pre-commit"
    hook_text = pre_commit.read_text(encoding="utf-8")
    assert hook_result["installed"]
    assert "nbdev-clean" in hook_text
    assert "nbdev-test" in hook_text
    assert (hook_root / ".git" / "info" / "nbskill-hooks-installed").exists()

    pre_commit.unlink()
    removed_result = install_nbdev_pre_commit_hooks(hook_root, run_nbdev_install_hooks=False)
    assert not removed_result["installed"]
    assert removed_result["reason"] == "hooks-removed-by-user"
    assert not pre_commit.exists()
    assert (hook_root / ".git" / "info" / "nbskill-hooks-disabled").exists()
finally:
    remove_demo_path(hook_root)


In [ ]:
#| hide
project_root = Path.cwd().parent
skill_text = (project_root / "nbskill" / "SKILL.md").read_text(encoding="utf-8")
template_text = (project_root / "nbskill" / "AGENTS.md").read_text(encoding="utf-8")
root_text = (project_root / "AGENTS.md").read_text(encoding="utf-8")
assert "nbskill.skill" in skill_text
assert "nbskill.skill" in template_text
assert "context" in root_text and "reference_query" in root_text
assert "prepare_change" not in root_text and "verify_change" not in root_text